<!--
Copyright (c) 2026 OceanBase.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 17 · 把现有技能包纳入管理

团队已有一个检查 CSV 的 Skill，除了入口说明，还有脚本和参考资料。我们先扫描它，按准确指纹导入，检查每个文件，再为项目生成一个 fork。

需要真实 Generation 模型用于 fork；原样 import 不改写包字节。所有文件位于本篇临时项目，不会扫描或修改你的个人 Skill 目录。

路线：准备完整包 → 扫描 → 原样导入与审核 → 下载校验 → 检测外部变更 → fork。

In [ ]:
import sys
from pathlib import Path

from _tutorial import Tutorial, show, table

from powercontext.http import CreateScopeRequest

if not Path("_tutorial.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "examples" / "jupyter"))
if previous_lab := globals().get("lab"):
    await previous_lab.close()
lab = await Tutorial.start("17", features=("generation",))
client = lab.client
assert client is not None
scope = await client.create_scope(
    CreateScopeRequest(
        title="订单 CSV 导入器 · 17", summary="本次教学实验的独立材料", idempotency_key=f"{lab.run_id}:main"
    )
)
scope_id = scope.scope_id

## 一个 Skill 可以带着可执行材料

这里写入一个很小、可以读完的检查脚本。脚本只检查 Decimal 的精度；参考资料说明范围。先运行脚本，再让 PowerContext 扫描这个明确配置的目录。

In [ ]:
import asyncio
import base64
import hashlib
import io
import zipfile

from powercontext.builtin.artifacts.skill.external import AgentSkillTarget
from powercontext.builtin.runtime.config import ExternalSkillsConfig

root = lab.directory / "external-skills"
package = root / "csv-amount-check"
(package / "scripts").mkdir(parents=True)
(package / "references").mkdir()
(package / "SKILL.md").write_text(
    "---\nname: csv-amount-check\ndescription: Check CSV amount precision before converting to cents.\n---\n\n# CSV amount check\nRun `python scripts/check.py`. Read references/limits.md before generalizing.\n",
    encoding="utf-8",
)
(package / "scripts" / "check.py").write_text(
    "from decimal import Decimal\nassert Decimal('1.999') * 100 != (Decimal('1.999') * 100).to_integral_value()\nprint('precision check passed')\n",
    encoding="utf-8",
)
(package / "scripts" / "check.py").chmod(0o755)
(package / "references" / "limits.md").write_text(
    "Only checks fractional cents. Does not validate currencies or import throughput.\n", encoding="utf-8"
)
process = await asyncio.create_subprocess_exec(
    sys.executable, str(package / "scripts" / "check.py"), stdout=asyncio.subprocess.PIPE
)
stdout, _ = await process.communicate()
assert process.returncode == 0
print(stdout.decode().strip())
lab.settings_overrides["external_skills"] = ExternalSkillsConfig(
    host_id=lab.run_id,
    targets=(
        AgentSkillTarget(target_id="tutorial-external", agent_kind="codex", installation_scope="project", path=root),
    ),
)
await lab.restart()
client = lab.client

## 扫描只登记，导入才进入审核

registration 的 fingerprint 指向本次看到的完整包。原样导入将这份快照变成候选，不会更新磁盘上的原件。

In [ ]:
from powercontext.http import (
    ApproveArtifactCandidateRequest,
    GetSkillPackageRequest,
    ImportExternalSkillRequest,
    ScanExternalSkillsRequest,
)

scan = await client.scan_external_skills(ScanExternalSkillsRequest(scope_id=scope_id))
assert len(scan.registrations) == 1
registration = scan.registrations[0]
show({"名称": registration.name, "指纹": registration.fingerprint})
imported = await client.import_external_skill(
    ImportExternalSkillRequest(
        scope_id=scope_id,
        external_skill_id=registration.external_skill_id,
        fingerprint=registration.fingerprint,
        mode="import",
        reason="纳入已逐文件检查的教学包",
    )
)
assert imported.candidate and imported.candidate.status == "pending"
show(imported.candidate.proposal)
approved = await client.approve_artifact_candidate(
    ApproveArtifactCandidateRequest(
        scope_id=scope_id, candidate_id=imported.candidate.candidate_id, expected_version=imported.candidate.version
    )
)
assert approved.result_artifact
artifact = approved.result_artifact

## 下载内容必须与导入时的每个文件一致

读取包清单，再下载 ZIP。我们逐个检查路径、摘要和字节；不会因为只有 SKILL.md 存在就声称完整包已保留。

In [ ]:
request = GetSkillPackageRequest(scope_id=scope_id, artifact=artifact)
manifest = await client.get_skill_package_manifest(request)
download = await client.download_skill_package(request)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(download.archive_base64))) as archive:
    rows = []
    for item in manifest.files:
        data = archive.read(item.path)
        assert data == (package / item.path).read_bytes()
        assert hashlib.sha256(data).hexdigest() == item.digest
        mode = archive.getinfo(item.path).external_attr >> 16
        assert bool(mode & 0o111) == item.executable
        assert item.executable == bool((package / item.path).stat().st_mode & 0o111)
        rows.append({"文件": item.path, "字节一致": True, "SHA-256": item.digest[:16]})
assert len(rows) == 3
table(rows)

## 外部文件变化后，旧指纹不能假装还是当前原件

修改参考资料，再解析旧 registration。旧 managed 版本应保持可下载；新的外部内容要重新扫描并明确选择。

In [ ]:
from powercontext.http import ResolveExternalSkillRequest

(package / "references" / "limits.md").write_text(
    "Also check non-finite values; do not claim throughput coverage.\n", encoding="utf-8"
)
stale = await client.resolve_external_skill(
    ResolveExternalSkillRequest(
        scope_id=scope_id, external_skill_id=registration.external_skill_id, fingerprint=registration.fingerprint
    )
)
assert stale.status == "unavailable"
new_scan = await client.scan_external_skills(ScanExternalSkillsRequest(scope_id=scope_id))
new_registration = new_scan.registrations[0]
assert new_registration.fingerprint != registration.fingerprint
assert (await client.download_skill_package(request)).package == download.package
show({"旧外部指纹": stale.status, "已导入的版本仍保留": True})

## fork 是新的模型建议，需要重新审核

这次要求模型把额外检查写进适用流程。名称必须符合标准技能包格式。生成内容保留原样，显示给读者检查；我们不自动采纳这份 fork。

In [ ]:
forked = await client.import_external_skill(
    ImportExternalSkillRequest(
        scope_id=scope_id,
        external_skill_id=new_registration.external_skill_id,
        fingerprint=new_registration.fingerprint,
        mode="fork",
        reason="Create a project-specific Skill named csv-amount-finite-check. Include explicit non-finite checks and preserve the limited validation scope; do not invent test results.",
    )
)
assert forked.candidate and forked.candidate.status == "pending"
assert forked.candidate.result_artifact is None
show(forked.candidate.proposal)

## 练习与验收

在 scripts 中修改一行，再扫描，预期指纹变化。比较 import 与 fork：前者逐文件保留，后者是待审核的新建议，不能默认承诺原脚本会被模型原样保留。

接下来阅读 [18_skill_distribution.ipynb](18_skill_distribution.ipynb)。

最后关闭服务。实验文件保留在本次 `.powercontext/` 目录，便于复查。

In [ ]:
await lab.close()
print("本篇 Server 已关闭。")